In [1]:
# Instalación de librerías
# Ejecutar al inicio de una sesión nueva de Colab.

!pip install -q faiss-cpu sentence-transformers pandas tqdm requests openpyxl python-docx networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 23.1 MB/s eta 0:00:00


In [2]:
# Cargar archivos manualmente
# Archivos necesarios:
# - indice_faiss_aml.index
# - chunks_filtrados.json
# - banco_evaluacion_con_docs_rel.json
# - grafo_aml_llm.json

from google.colab import files
import os

print("Selecciona los cuatro archivos necesarios:")
uploaded = files.upload()

print("\nArchivos cargados:")
for nombre in uploaded.keys():
    print("-", nombre)

Selecciona los cuatro archivos necesarios:


Saving banco_evaluacion_con_docs_rel.json to banco_evaluacion_con_docs_rel.json
Saving chunks_filtrados.json to chunks_filtrados.json
Saving grafo_aml_llm.json to grafo_aml_llm.json
Saving indice_faiss_aml.index to indice_faiss_aml.index

Archivos cargados:
- banco_evaluacion_con_docs_rel.json
- chunks_filtrados.json
- grafo_aml_llm.json
- indice_faiss_aml.index


In [3]:
# Definir rutas de trabajo y parámetros globales del experimento

import os

RUTA_BASE = "/content"

RUTA_INDICE_CHUNKS = os.path.join(RUTA_BASE, "indice_faiss_aml.index")
RUTA_CHUNKS = os.path.join(RUTA_BASE, "chunks_filtrados.json")
RUTA_BANCO = os.path.join(RUTA_BASE, "banco_evaluacion_con_docs_rel.json")
RUTA_GRAFO = os.path.join(RUTA_BASE, "grafo_aml_llm.json")

RUTA_UNIDADES_GRAFO = os.path.join(RUTA_BASE, "unidades_grafo_recuperables.json")
RUTA_INDICE_GRAFO = os.path.join(RUTA_BASE, "indice_faiss_grafo.index")

RUTA_SALIDA_JSON = os.path.join(RUTA_BASE, "resultados_rag_grafos.json")
RUTA_SALIDA_CSV = os.path.join(RUTA_BASE, "resultados_rag_grafos.csv")
RUTA_RESULTADOS_RAG_GRAFOS_PARCIAL = os.path.join(RUTA_BASE, "resultados_rag_grafos_parcial.json")

# Parámetros principales de recuperación. Estos son los valores usados en la ejecución final.
TOP_K_CHUNKS = 5
TOP_K_GRAFO = 5
PROFUNDIDAD_GRAFO = 1

MAX_UNIDADES_GRAFO = 5
MAX_NODOS_ACTIVADOS = 8
MAX_NODOS_VECINOS = 6
MAX_ARISTAS_VECINAS = 10

# Parámetros de generación. num_ctx es la ventana total; num_predict reserva salida máxima.
MODELO_GENERATIVO = "llama3.2:3b"
NUM_CTX_GENERACION = 8192
NUM_PREDICT_GENERACION = 512
TIMEOUT_GENERACION = 300

# Presupuesto de contexto para evitar que el prompt consuma toda la ventana de tokens del modelo.
# La ejecución anterior falló porque el prompt llegó a ocupar casi 8192 tokens y dejó 1 token de salida.
MAX_CARACTERES_CHUNK_GENERACION = 1100
MAX_CARACTERES_UNIDAD_GRAFO_GENERACION = 550
MAX_CARACTERES_NODO_GRAFO_GENERACION = 450
MAX_CARACTERES_ARISTA_GRAFO_GENERACION = 550
MAX_CARACTERES_CONTEXTO_GENERACION = 15000
MAX_LONGITUD_PROMPT_GENERACION = 20000
MAX_CARACTERES_CONTEXTO_REINTENTO = 9000

rutas_necesarias = [
    RUTA_INDICE_CHUNKS,
    RUTA_CHUNKS,
    RUTA_BANCO,
    RUTA_GRAFO
]

for ruta in rutas_necesarias:
    print(ruta, "->", os.path.exists(ruta))

faltantes = [ruta for ruta in rutas_necesarias if not os.path.exists(ruta)]
if faltantes:
    raise FileNotFoundError(
        "Faltan archivos necesarios. Revisa que los nombres coincidan exactamente: "
        + ", ".join(faltantes)
    )


/content/indice_faiss_aml.index -> True
/content/chunks_filtrados.json -> True
/content/banco_evaluacion_con_docs_rel.json -> True
/content/grafo_aml_llm.json -> True


In [4]:
# Cargar índice, chunks, banco de evaluación y grafo

import json
import faiss

indice_chunks = faiss.read_index(RUTA_INDICE_CHUNKS)

with open(RUTA_CHUNKS, "r", encoding="utf-8") as f:
    chunks = json.load(f)

with open(RUTA_BANCO, "r", encoding="utf-8") as f:
    banco = json.load(f)

with open(RUTA_GRAFO, "r", encoding="utf-8") as f:
    grafo_json = json.load(f)

nodos_grafo = grafo_json.get("nodes", [])
aristas_grafo = grafo_json.get("edges", [])
metadata_grafo = grafo_json.get("metadata", {})

print("Vectores en FAISS de chunks:", indice_chunks.ntotal)
print("Chunks cargados:", len(chunks))
print("Preguntas cargadas:", len(banco))
print("Nodos del grafo:", len(nodos_grafo))
print("Aristas del grafo:", len(aristas_grafo))
print("Metadatos del grafo:", metadata_grafo)

if indice_chunks.ntotal != len(chunks):
    raise ValueError(
        f"El índice FAISS contiene {indice_chunks.ntotal} vectores, pero se han cargado {len(chunks)} chunks. "
        "Debes usar el índice y el JSON de chunks generados en la misma vectorización."
    )

if len(banco) == 0:
    raise ValueError("El banco de evaluación está vacío.")

if len(nodos_grafo) == 0:
    raise ValueError("El grafo no contiene nodos.")

if len(aristas_grafo) == 0:
    raise ValueError("El grafo no contiene aristas.")

Vectores en FAISS de chunks: 1529
Chunks cargados: 1529
Preguntas cargadas: 30
Nodos del grafo: 109
Aristas del grafo: 119
Metadatos del grafo: {'nombre_grafo': 'grafo_aml_llm', 'descripcion': 'Grafo de conocimiento AML/CFT construido mediante extracción asistida por LLM a partir del corpus documental.', 'idioma': 'es', 'metodo_extraccion': 'LLM con ontología controlada', 'total_nodos': 109, 'total_aristas': 119}


In [5]:
# Revisar estructura de los archivos

print("Ejemplo de chunk:")
print(json.dumps(chunks[0], ensure_ascii=False, indent=2)[:1500])

print("\nEjemplo de pregunta:")
print(json.dumps(banco[0], ensure_ascii=False, indent=2)[:1500])

print("\nEjemplo de nodo:")
print(json.dumps(nodos_grafo[0], ensure_ascii=False, indent=2)[:1500])

print("\nEjemplo de arista:")
print(json.dumps(aristas_grafo[0], ensure_ascii=False, indent=2)[:1500])

Ejemplo de chunk:
{
  "chunk_id": "001_0001",
  "doc_id": "001",
  "titulo": "Lavado de dinero",
  "categoria": "Tema central",
  "fuente": "Wikipedia",
  "idioma": "es",
  "tipo_documento": "entrada_enciclopedica",
  "formato_original": "html",
  "url": "https://es.wikipedia.org/wiki/Lavado_de_dinero",
  "texto_chunk": "El lavado de dinero (también conocido como lavado de capitales, lavado de activos, blanqueo de dinero,​ o blanqueo de capital) es el proceso de cambiar grandes cantidades de dinero obtenidas de delitos, como el narcotráfico, por otras de origen legítimo.​​ Es un delito en muchas jurisdicciones con definiciones diversas. Es una operación esencial del crimen organizado y de la economía sumergida. El blanqueo de capitales comienza con la comisión de un acto delictivo de tipo grave (tal como la obtención de beneficios ilegales en los mercados financieros u otros sectores económicos). El blanqueo de capitales es un delito autónomo que no requiere de una condena judicial pre

In [6]:
# Cargar modelo de embeddings E5

from sentence_transformers import SentenceTransformer
import numpy as np

MODELO_EMBEDDINGS = "intfloat/multilingual-e5-base"

modelo_embeddings = SentenceTransformer(MODELO_EMBEDDINGS)

print("Modelo cargado:", MODELO_EMBEDDINGS)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Modelo cargado: intfloat/multilingual-e5-base


In [7]:
# Funciones de vectorización

def vectorizar_pregunta(pregunta):
    texto = "query: " + str(pregunta)

    embedding = modelo_embeddings.encode(
        [texto],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    return embedding.astype("float32")


def vectorizar_pasajes(lista_textos, batch_size=32):
    textos = ["passage: " + str(t) for t in lista_textos]

    embeddings = modelo_embeddings.encode(
        textos,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    return embeddings.astype("float32")

In [8]:
# Funciones de recuperación vectorial de chunks

def obtener_texto_chunk(chunk):
    posibles_claves = [
        "texto_chunk",
        "texto",
        "text",
        "content",
        "contenido"
    ]

    for clave in posibles_claves:
        if clave in chunk and chunk[clave]:
            return chunk[clave]

    return ""


def recuperar_chunks(pregunta, top_k=5):
    embedding_pregunta = vectorizar_pregunta(pregunta)

    scores, indices = indice_chunks.search(embedding_pregunta, top_k)

    resultados = []

    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        if int(idx) < 0:
            continue

        chunk = chunks[int(idx)]
        texto = obtener_texto_chunk(chunk)

        resultados.append({
            "rank": rank,
            "indice_chunk": int(idx),
            "score": float(score),
            "texto": texto,
            "metadatos": chunk
        })

    return resultados

In [9]:
# Construcción de estructuras auxiliares del grafo

from collections import defaultdict
import networkx as nx

nodos_por_id = {n["id"]: n for n in nodos_grafo}

# Grafo dirigido: las aristas conservan source -> target.
G = nx.DiGraph()

for nodo in nodos_grafo:
    G.add_node(
        nodo["id"],
        label=nodo.get("label", ""),
        type=nodo.get("type", ""),
        data=nodo
    )

for arista in aristas_grafo:
    source = arista.get("source")
    target = arista.get("target")
    if source in nodos_por_id and target in nodos_por_id:
        G.add_edge(
            source,
            target,
            relation=arista.get("relation", ""),
            data=arista
        )

aristas_por_id = {a["id"]: a for a in aristas_grafo}
aristas_salientes = defaultdict(list)
aristas_entrantes = defaultdict(list)

for arista in aristas_grafo:
    source = arista.get("source")
    target = arista.get("target")
    aristas_salientes[source].append(arista)
    aristas_entrantes[target].append(arista)

print("Nodos en NetworkX:", G.number_of_nodes())
print("Aristas en NetworkX:", G.number_of_edges())

Nodos en NetworkX: 109
Aristas en NetworkX: 119


Construcción de estructuras auxiliares del grafo:

En la celda anterior se transforma el grafo almacenado en el archivo JSON en una estructura operativa con `NetworkX`. Para ello, primero se crea un diccionario de nodos por identificador (`nodos_por_id`), que permite acceder rápidamente a la información de cada nodo. A continuación, se construye un grafo dirigido (`DiGraph`), ya que las relaciones del grafo conservan una orientación entre nodo de origen y nodo de destino.

Después, se añaden al grafo todos los nodos y únicamente aquellas aristas cuyos nodos de origen y destino existen en el diccionario de nodos, evitando así posibles relaciones huérfanas. También se generan diccionarios auxiliares de aristas salientes y entrantes, que permiten recuperar de forma rápida las relaciones conectadas con un nodo determinado. Estas estructuras serán necesarias más adelante para realizar la expansión por vecinos del grafo durante la recuperación.

La salida de la celda muestra el número total de nodos y aristas cargados en `NetworkX`, lo que permite comprobar que la estructura del grafo se ha construido correctamente a partir del JSON original.


In [10]:
# Funciones auxiliares para evidencias y normalización textual

import re
import unicodedata

def normalizar_texto_busqueda(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    texto = re.sub(r"[^a-z0-9ñáéíóúü\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def extraer_snippets(evidencias, max_evidencias=2, max_caracteres=900):
    if not evidencias:
        return ""

    partes = []

    for ev in evidencias[:max_evidencias]:
        titulo = ev.get("titulo", "")
        fuente = ev.get("fuente", "")
        doc_id = ev.get("doc_id", "")
        snippet = ev.get("snippet", "")

        texto = ""
        if doc_id or titulo or fuente:
            texto += f"[doc_id={doc_id} | título={titulo} | fuente={fuente}]\n"
        texto += str(snippet)

        if len(texto) > max_caracteres:
            texto = texto[:max_caracteres] + "\n[Evidencia truncada]"

        partes.append(texto)

    return "\n".join(partes)


def etiqueta_o_alias_en_texto(etiqueta, texto_normalizado):
    etiqueta_norm = normalizar_texto_busqueda(etiqueta)

    # Evita falsos positivos con alias demasiado breves, salvo casos muy reconocibles.
    if len(etiqueta_norm) < 4 and etiqueta_norm not in {"ue", "onu", "gafi", "ofac", "bis"}:
        return False

    patron = r"\b" + re.escape(etiqueta_norm) + r"\b"
    return re.search(patron, texto_normalizado) is not None

In [11]:
# Crear unidades recuperables del grafo
# Se crean tres tipos de unidades:
# - nodo
# - arista
# - evidencia
#
# Estas unidades se vectorizan para permitir recuperación semántica sobre el grafo.

def crear_texto_unidad_nodo(nodo):
    aliases = ", ".join(nodo.get("aliases", []) or [])
    evidencia = extraer_snippets(nodo.get("evidence", []), max_evidencias=2)

    texto = (
        f"Nodo del grafo AML/CFT. "
        f"Entidad: {nodo.get('label', '')}. "
        f"Tipo: {nodo.get('type', '')}. "
        f"Alias: {aliases}. "
        f"Descripción: {nodo.get('description', '')}. "
        f"Evidencia documental: {evidencia}"
    )

    return texto.strip()


def crear_texto_unidad_arista(arista):
    source_id = arista.get("source", "")
    target_id = arista.get("target", "")
    source_label = nodos_por_id.get(source_id, {}).get("label", source_id)
    target_label = nodos_por_id.get(target_id, {}).get("label", target_id)
    evidencia = extraer_snippets(arista.get("evidence", []), max_evidencias=2)

    texto = (
        f"Relación del grafo AML/CFT. "
        f"{source_label} --{arista.get('relation', '')}--> {target_label}. "
        f"Descripción: {arista.get('description', '')}. "
        f"Confianza: {arista.get('confidence', '')}. "
        f"Evidencia documental: {evidencia}"
    )

    return texto.strip()


def crear_unidades_recuperables_grafo(nodos, aristas):
    unidades = []

    for nodo in nodos:
        unidades.append({
            "id": f"node::{nodo.get('id')}",
            "tipo_unidad": "nodo",
            "node_id": nodo.get("id"),
            "edge_id": None,
            "source": None,
            "target": None,
            "texto": crear_texto_unidad_nodo(nodo),
            "metadatos": {
                "tipo_unidad": "nodo",
                "node_id": nodo.get("id"),
                "label": nodo.get("label", ""),
                "type": nodo.get("type", ""),
                "aliases": nodo.get("aliases", []),
            }
        })

        for i, ev in enumerate(nodo.get("evidence", []) or [], start=1):
            unidades.append({
                "id": f"evidence::node::{nodo.get('id')}::{i}",
                "tipo_unidad": "evidencia_nodo",
                "node_id": nodo.get("id"),
                "edge_id": None,
                "source": None,
                "target": None,
                "texto": (
                    f"Evidencia asociada al nodo {nodo.get('label', '')}. "
                    f"Tipo: {nodo.get('type', '')}. "
                    f"Fragmento: {ev.get('snippet', '')}"
                ),
                "metadatos": {
                    "tipo_unidad": "evidencia_nodo",
                    "node_id": nodo.get("id"),
                    "label": nodo.get("label", ""),
                    "doc_id": ev.get("doc_id", ""),
                    "titulo": ev.get("titulo", ""),
                    "fuente": ev.get("fuente", ""),
                    "url": ev.get("url", "")
                }
            })

    for arista in aristas:
        source_id = arista.get("source", "")
        target_id = arista.get("target", "")
        source_label = nodos_por_id.get(source_id, {}).get("label", source_id)
        target_label = nodos_por_id.get(target_id, {}).get("label", target_id)

        unidades.append({
            "id": f"edge::{arista.get('id')}",
            "tipo_unidad": "arista",
            "node_id": None,
            "edge_id": arista.get("id"),
            "source": source_id,
            "target": target_id,
            "texto": crear_texto_unidad_arista(arista),
            "metadatos": {
                "tipo_unidad": "arista",
                "edge_id": arista.get("id"),
                "source": source_id,
                "target": target_id,
                "source_label": source_label,
                "target_label": target_label,
                "relation": arista.get("relation", ""),
                "confidence": arista.get("confidence", "")
            }
        })

        for i, ev in enumerate(arista.get("evidence", []) or [], start=1):
            unidades.append({
                "id": f"evidence::edge::{arista.get('id')}::{i}",
                "tipo_unidad": "evidencia_arista",
                "node_id": None,
                "edge_id": arista.get("id"),
                "source": source_id,
                "target": target_id,
                "texto": (
                    f"Evidencia asociada a la relación {source_label} --{arista.get('relation', '')}--> {target_label}. "
                    f"Descripción: {arista.get('description', '')}. "
                    f"Fragmento: {ev.get('snippet', '')}"
                ),
                "metadatos": {
                    "tipo_unidad": "evidencia_arista",
                    "edge_id": arista.get("id"),
                    "source": source_id,
                    "target": target_id,
                    "source_label": source_label,
                    "target_label": target_label,
                    "relation": arista.get("relation", ""),
                    "doc_id": ev.get("doc_id", ""),
                    "titulo": ev.get("titulo", ""),
                    "fuente": ev.get("fuente", ""),
                    "url": ev.get("url", "")
                }
            })

    return unidades


unidades_grafo = crear_unidades_recuperables_grafo(nodos_grafo, aristas_grafo)

print("Unidades recuperables del grafo:", len(unidades_grafo))
print(json.dumps(unidades_grafo[0], ensure_ascii=False, indent=2)[:1500])

Unidades recuperables del grafo: 456
{
  "id": "node::abstencion_de_ejecucion",
  "tipo_unidad": "nodo",
  "node_id": "abstencion_de_ejecucion",
  "edge_id": null,
  "source": null,
  "target": null,
  "texto": "Nodo del grafo AML/CFT. Entidad: Abstención de ejecución. Tipo: obligacion_normativa. Alias: . Descripción: Obligación prevista junto a la comunicación por indicio para no ejecutar determinadas operaciones en supuestos establecidos.. Evidencia documental: [doc_id=102 | título=Ley 10/2010, de 28 de abril, de prevención del blanqueo de capitales y de la financiación del terrorismo | fuente=SEPBLAC]\n…en que, habiendo sido puestas de manifiesto internamente, el sujeto obligado no hubiese informado al directivo o empleado comunicante del curso dado a su comunicación. Artículo 19. Abstención de ejecución. 1. Los sujetos obligados se abstendrán de ejecutar cualquier operación de las señaladas en el artículo precedente. No obstante, cuando dicha abstención no sea posible o pueda dific

In [12]:
# Vectorizar unidades recuperables del grafo y crear índice FAISS propio

textos_unidades_grafo = [u["texto"] for u in unidades_grafo]

embeddings_grafo = vectorizar_pasajes(textos_unidades_grafo, batch_size=32)

dim = embeddings_grafo.shape[1]
indice_grafo = faiss.IndexFlatIP(dim)
indice_grafo.add(embeddings_grafo)

print("Dimensión embeddings grafo:", dim)
print("Vectores en índice FAISS del grafo:", indice_grafo.ntotal)

# Guardar índice y unidades para trazabilidad
faiss.write_index(indice_grafo, RUTA_INDICE_GRAFO)

with open(RUTA_UNIDADES_GRAFO, "w", encoding="utf-8") as f:
    json.dump(unidades_grafo, f, ensure_ascii=False, indent=2)

print("Índice del grafo guardado en:", RUTA_INDICE_GRAFO)
print("Unidades recuperables guardadas en:", RUTA_UNIDADES_GRAFO)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Dimensión embeddings grafo: 768
Vectores en índice FAISS del grafo: 456
Índice del grafo guardado en: /content/indice_faiss_grafo.index
Unidades recuperables guardadas en: /content/unidades_grafo_recuperables.json


In [13]:
# Recuperación semántica de unidades del grafo

def recuperar_unidades_grafo(pregunta, top_k=None):
    if top_k is None:
        top_k = globals().get("TOP_K_GRAFO", 5)

    embedding_pregunta = vectorizar_pregunta(pregunta)
    scores, indices = indice_grafo.search(embedding_pregunta, top_k)

    resultados = []

    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        if int(idx) < 0:
            continue

        unidad = unidades_grafo[int(idx)]

        resultados.append({
            "rank": rank,
            "indice_unidad": int(idx),
            "score": float(score),
            "id_unidad": unidad.get("id"),
            "tipo_unidad": unidad.get("tipo_unidad"),
            "node_id": unidad.get("node_id"),
            "edge_id": unidad.get("edge_id"),
            "source": unidad.get("source"),
            "target": unidad.get("target"),
            "texto": unidad.get("texto", ""),
            "metadatos": unidad.get("metadatos", {})
        })

    return resultados


In [14]:
# Localización de entidades del grafo en preguntas y chunks

def localizar_entidades_en_texto(texto, max_entidades=20):
    texto_norm = normalizar_texto_busqueda(texto)
    entidades = []

    for nodo in nodos_grafo:
        candidatos = [nodo.get("label", "")]
        candidatos.extend(nodo.get("aliases", []) or [])

        encontrado_por = None

        for candidato in candidatos:
            if candidato and etiqueta_o_alias_en_texto(candidato, texto_norm):
                encontrado_por = candidato
                break

        if encontrado_por:
            entidades.append({
                "node_id": nodo.get("id"),
                "label": nodo.get("label", ""),
                "type": nodo.get("type", ""),
                "match": encontrado_por
            })

    # Eliminar duplicados conservando orden
    vistos = set()
    entidades_unicas = []

    for e in entidades:
        if e["node_id"] not in vistos:
            entidades_unicas.append(e)
            vistos.add(e["node_id"])

    return entidades_unicas[:max_entidades]


def localizar_entidades_en_chunks(chunks_recuperados, max_entidades_total=30):
    entidades = []
    vistos = set()

    for chunk in chunks_recuperados:
        entidades_chunk = localizar_entidades_en_texto(
            chunk.get("texto", ""),
            max_entidades=max_entidades_total
        )

        for e in entidades_chunk:
            if e["node_id"] not in vistos:
                entidades.append({
                    **e,
                    "rank_chunk_origen": chunk.get("rank"),
                    "score_chunk_origen": chunk.get("score")
                })
                vistos.add(e["node_id"])

        if len(entidades) >= max_entidades_total:
            break

    return entidades[:max_entidades_total]

In [15]:
# Activación de nodos y expansión por vecinos

def anadir_unico(lista, valor):
    """Añade un valor a una lista solo si existe y no estaba ya presente."""
    if valor and valor in nodos_por_id and valor not in lista:
        lista.append(valor)


def nodos_desde_unidades_grafo(unidades_recuperadas):
    """
    Extrae nodos a partir de las unidades recuperadas del grafo conservando el orden.
    Se incluyen node_id, source, target y los extremos de las aristas asociadas.
    """
    nodos = []

    for unidad in unidades_recuperadas:
        anadir_unico(nodos, unidad.get("node_id"))
        anadir_unico(nodos, unidad.get("source"))
        anadir_unico(nodos, unidad.get("target"))

        edge_id = unidad.get("edge_id")
        if edge_id and edge_id in aristas_por_id:
            arista = aristas_por_id[edge_id]
            anadir_unico(nodos, arista.get("source"))
            anadir_unico(nodos, arista.get("target"))

    return nodos


def activar_nodos(pregunta, chunks_recuperados, unidades_grafo_recuperadas, max_nodos_activados=None):
    """
    Activa nodos del grafo a partir de tres señales:
    1. entidades mencionadas en la pregunta;
    2. entidades localizadas en los chunks recuperados;
    3. nodos derivados de las unidades del grafo recuperadas semánticamente.

    Se conserva un orden determinista. Primero se priorizan las entidades de la pregunta,
    después las unidades del grafo y finalmente las entidades detectadas en los chunks.
    """
    entidades_pregunta = localizar_entidades_en_texto(pregunta, max_entidades=20)
    entidades_chunks = localizar_entidades_en_chunks(chunks_recuperados, max_entidades_total=30)
    nodos_unidades = nodos_desde_unidades_grafo(unidades_grafo_recuperadas)

    nodos_detectados = []

    # Prioridad 1: lo que aparece explícitamente en la pregunta.
    for e in entidades_pregunta:
        anadir_unico(nodos_detectados, e.get("node_id"))

    # Prioridad 2: lo recuperado semánticamente del grafo.
    for nodo_id in nodos_unidades:
        anadir_unico(nodos_detectados, nodo_id)

    # Prioridad 3: entidades que aparecen en los chunks recuperados.
    for e in entidades_chunks:
        anadir_unico(nodos_detectados, e.get("node_id"))

    if max_nodos_activados is None:
        nodos_usados = nodos_detectados
    else:
        nodos_usados = nodos_detectados[:max_nodos_activados]

    return {
        "entidades_pregunta": entidades_pregunta,
        "entidades_chunks": entidades_chunks,
        "nodos_activados_detectados": nodos_detectados,
        "nodos_activados": nodos_usados
    }


def expandir_por_vecinos(nodos_activados, profundidad=1, max_nodos_vecinos=None, max_aristas=None):
    """
    Expande los nodos activados por vecinos entrantes y salientes.

    Importante: max_nodos_vecinos limita solo los vecinos nuevos añadidos,
    no la suma de nodos activados + vecinos. Así se evita que, si ya hay 8 nodos
    activados, la expansión quede bloqueada por un máximo de 6 vecinos.
    """
    if max_nodos_vecinos is None:
        max_nodos_vecinos = globals().get("MAX_NODOS_VECINOS", 6)
    if max_aristas is None:
        max_aristas = globals().get("MAX_ARISTAS_VECINAS", 10)

    nodos_activados = [n for n in nodos_activados if n in nodos_por_id]

    visitados = set(nodos_activados)
    vecinos_nuevos = []
    frontera = list(nodos_activados)
    aristas_usadas = []
    aristas_vistas = set()

    for _ in range(profundidad):
        nueva_frontera = []

        for nodo_id in list(frontera):
            candidatas = []
            candidatas.extend(aristas_salientes.get(nodo_id, []))
            candidatas.extend(aristas_entrantes.get(nodo_id, []))

            for arista in candidatas:
                if len(aristas_usadas) >= max_aristas or len(vecinos_nuevos) >= max_nodos_vecinos:
                    break

                aid = arista.get("id")
                if aid not in aristas_vistas:
                    aristas_usadas.append(arista)
                    aristas_vistas.add(aid)

                source = arista.get("source")
                target = arista.get("target")

                for vecino in [source, target]:
                    if (
                        vecino
                        and vecino in nodos_por_id
                        and vecino not in visitados
                        and len(vecinos_nuevos) < max_nodos_vecinos
                    ):
                        visitados.add(vecino)
                        vecinos_nuevos.append(vecino)
                        nueva_frontera.append(vecino)

            if len(aristas_usadas) >= max_aristas or len(vecinos_nuevos) >= max_nodos_vecinos:
                break

        frontera = nueva_frontera

        if not frontera:
            break

    return vecinos_nuevos[:max_nodos_vecinos], aristas_usadas[:max_aristas]


In [16]:
# Construcción del contexto final para el generador y el evaluador

from collections import Counter

def limitar_texto_generacion(texto, max_caracteres=None, marca="[Texto truncado]"):
    """Limita un texto para controlar el tamaño del prompt de generación."""
    if max_caracteres is None:
        max_caracteres = globals().get("MAX_CARACTERES_CHUNK_GENERACION", 1100)
    if texto is None:
        return ""
    texto = str(texto).strip()
    if len(texto) <= max_caracteres:
        return texto
    return texto[:max_caracteres].rstrip() + f"\n{marca}"


def extraer_metadatos_basicos(metadatos):
    """Reduce metadatos para evitar que el prompt se llene con información auxiliar."""
    metadatos = metadatos or {}
    return {
        "doc_id": metadatos.get("doc_id", ""),
        "chunk_id": metadatos.get("chunk_id", ""),
        "titulo": metadatos.get("titulo", ""),
        "fuente": metadatos.get("fuente", ""),
        "url": metadatos.get("url", "")
    }


def metadatos_desde_evidencia(evidencias):
    if evidencias:
        ev = evidencias[0]
        return {
            "doc_id": ev.get("doc_id", ""),
            "chunk_id": "",
            "titulo": ev.get("titulo", ""),
            "fuente": ev.get("fuente", ""),
            "url": ev.get("url", "")
        }

    return {"doc_id": "", "chunk_id": "", "titulo": "", "fuente": "", "url": ""}


def texto_nodo_para_contexto(nodo_id, max_caracteres=None):
    if max_caracteres is None:
        max_caracteres = globals().get("MAX_CARACTERES_NODO_GRAFO_GENERACION", 450)

    nodo = nodos_por_id.get(nodo_id, {})
    aliases = ", ".join(nodo.get("aliases", []) or [])
    evidencia = extraer_snippets(nodo.get("evidence", []), max_evidencias=1, max_caracteres=250)

    texto = (
        f"[Nodo del grafo]\n"
        f"Entidad: {nodo.get('label', '')}\n"
        f"Tipo: {nodo.get('type', '')}\n"
        f"Alias: {aliases}\n"
        f"Descripción: {nodo.get('description', '')}\n"
        f"Evidencia: {evidencia}"
    )

    return limitar_texto_generacion(texto, max_caracteres=max_caracteres)


def texto_arista_para_contexto(arista, max_caracteres=None):
    if max_caracteres is None:
        max_caracteres = globals().get("MAX_CARACTERES_ARISTA_GRAFO_GENERACION", 550)

    source = arista.get("source", "")
    target = arista.get("target", "")
    source_label = nodos_por_id.get(source, {}).get("label", source)
    target_label = nodos_por_id.get(target, {}).get("label", target)
    evidencia = extraer_snippets(arista.get("evidence", []), max_evidencias=1, max_caracteres=250)

    texto = (
        f"[Relación del grafo]\n"
        f"{source_label} --{arista.get('relation', '')}--> {target_label}\n"
        f"Descripción: {arista.get('description', '')}\n"
        f"Confianza: {arista.get('confidence', '')}\n"
        f"Evidencia: {evidencia}"
    )

    return limitar_texto_generacion(texto, max_caracteres=max_caracteres)


def construir_contexto_final(
    chunks_recuperados,
    unidades_grafo_recuperadas,
    nodos_activados,
    nodos_vecinos,
    aristas_vecinas,
    max_unidades_grafo=None,
    max_nodos_activados=None,
    max_nodos_vecinos=None,
    max_aristas_vecinas=None,
    presupuesto_contexto=None
):
    """
    Construye el contexto final que recibirá el modelo generador.

    La función no solo limita cada fragmento por separado, sino también el tamaño total
    del contexto. Esto evita que el prompt ocupe toda la ventana `num_ctx` de Ollama y
    deje al modelo sin espacio para generar la respuesta.
    """
    if max_unidades_grafo is None:
        max_unidades_grafo = globals().get("MAX_UNIDADES_GRAFO", 5)
    if max_nodos_activados is None:
        max_nodos_activados = globals().get("MAX_NODOS_ACTIVADOS", 8)
    if max_nodos_vecinos is None:
        max_nodos_vecinos = globals().get("MAX_NODOS_VECINOS", 6)
    if max_aristas_vecinas is None:
        max_aristas_vecinas = globals().get("MAX_ARISTAS_VECINAS", 10)
    if presupuesto_contexto is None:
        presupuesto_contexto = globals().get("MAX_CARACTERES_CONTEXTO_GENERACION", 15000)

    contexto_final = []
    rank = 1
    caracteres_usados = 0

    def anadir_fragmento(texto, score, metadatos, max_caracteres):
        nonlocal rank, caracteres_usados

        texto = limitar_texto_generacion(texto, max_caracteres=max_caracteres)
        restante = presupuesto_contexto - caracteres_usados

        # Si queda muy poco presupuesto, no se añade más contexto.
        if restante <= 250:
            return False

        if len(texto) > restante:
            texto = limitar_texto_generacion(texto, max_caracteres=restante)

        contexto_final.append({
            "rank": rank,
            "score": score,
            "texto": texto,
            "metadatos": metadatos
        })
        rank += 1
        caracteres_usados += len(texto)
        return True

    # 1. Chunks recuperados por FAISS. Se priorizan porque contienen evidencia textual directa.
    for c in chunks_recuperados:
        metadatos = {
            **extraer_metadatos_basicos(c.get("metadatos", {})),
            "tipo_contexto": "chunk_faiss",
            "indice_chunk": c.get("indice_chunk")
        }
        anadir_fragmento(
            texto=c.get("texto", ""),
            score=c.get("score", 0.0),
            metadatos=metadatos,
            max_caracteres=globals().get("MAX_CARACTERES_CHUNK_GENERACION", 1100)
        )

    # 2. Unidades del grafo recuperadas semánticamente.
    for u in unidades_grafo_recuperadas[:max_unidades_grafo]:
        metadatos = {
            **u.get("metadatos", {}),
            "tipo_contexto": "unidad_grafo",
            "id_unidad": u.get("id_unidad"),
            "tipo_unidad": u.get("tipo_unidad")
        }
        anadir_fragmento(
            texto="[Unidad recuperable del grafo]\n" + u.get("texto", ""),
            score=u.get("score", 0.0),
            metadatos=metadatos,
            max_caracteres=globals().get("MAX_CARACTERES_UNIDAD_GRAFO_GENERACION", 550)
        )

    # 3. Nodos activados.
    for nodo_id in nodos_activados[:max_nodos_activados]:
        nodo = nodos_por_id.get(nodo_id, {})
        metadatos = {
            **metadatos_desde_evidencia(nodo.get("evidence", [])),
            "tipo_contexto": "nodo_activado",
            "node_id": nodo_id,
            "label": nodo.get("label", ""),
            "type": nodo.get("type", "")
        }
        anadir_fragmento(
            texto=texto_nodo_para_contexto(nodo_id),
            score=0.0,
            metadatos=metadatos,
            max_caracteres=globals().get("MAX_CARACTERES_NODO_GRAFO_GENERACION", 450)
        )

    # 4. Relaciones vecinas tras expansión.
    for arista in aristas_vecinas[:max_aristas_vecinas]:
        metadatos = {
            **metadatos_desde_evidencia(arista.get("evidence", [])),
            "tipo_contexto": "arista_vecina",
            "edge_id": arista.get("id"),
            "source": arista.get("source"),
            "target": arista.get("target"),
            "relation": arista.get("relation", "")
        }
        anadir_fragmento(
            texto=texto_arista_para_contexto(arista),
            score=0.0,
            metadatos=metadatos,
            max_caracteres=globals().get("MAX_CARACTERES_ARISTA_GRAFO_GENERACION", 550)
        )

    # 5. Nodos vecinos.
    for nodo_id in nodos_vecinos[:max_nodos_vecinos]:
        nodo = nodos_por_id.get(nodo_id, {})
        metadatos = {
            **metadatos_desde_evidencia(nodo.get("evidence", [])),
            "tipo_contexto": "nodo_vecino",
            "node_id": nodo_id,
            "label": nodo.get("label", ""),
            "type": nodo.get("type", "")
        }
        anadir_fragmento(
            texto=texto_nodo_para_contexto(nodo_id),
            score=0.0,
            metadatos=metadatos,
            max_caracteres=globals().get("MAX_CARACTERES_NODO_GRAFO_GENERACION", 450)
        )

    return contexto_final


def recortar_contexto_final_por_presupuesto(contexto_final, max_caracteres):
    """
    Recorta un contexto ya construido para reintentos cuando el prompt queda demasiado largo.
    Mantiene el orden original y conserva prioritariamente los primeros elementos.
    """
    contexto_recortado = []
    usados = 0

    for r in contexto_final:
        texto = str(r.get("texto", ""))
        restante = max_caracteres - usados
        if restante <= 250:
            break

        nuevo = dict(r)
        if len(texto) > restante:
            nuevo["texto"] = limitar_texto_generacion(texto, max_caracteres=restante)
        contexto_recortado.append(nuevo)
        usados += len(nuevo.get("texto", ""))

    # Recalcular ranks para mantener coherencia.
    for i, r in enumerate(contexto_recortado, start=1):
        r["rank"] = i

    return contexto_recortado


def conteo_tipos_contexto(contexto_final):
    return Counter(
        r.get("metadatos", {}).get("tipo_contexto", "sin_tipo")
        for r in contexto_final
    )


In [17]:
# Prompt para usar con el sistema RAG de grafos
# El contexto combina chunks, unidades del grafo, nodos y aristas.
# Se mantiene un prompt breve para reservar espacio de generación dentro de num_ctx.

def construir_prompt(pregunta, contexto_final):
    partes_contexto = []

    for r in contexto_final:
        metadatos = r.get("metadatos", {})
        doc_id = metadatos.get("doc_id", "")
        chunk_id = metadatos.get("chunk_id", "")
        titulo = metadatos.get("titulo", "")
        fuente = metadatos.get("fuente", "")
        tipo_contexto = metadatos.get("tipo_contexto", "")

        cabecera = (
            f"[Fragmento {r.get('rank', '')} | "
            f"tipo={tipo_contexto} | "
            f"doc_id={doc_id} | "
            f"chunk_id={chunk_id}]"
        )

        bloque = cabecera + "\n"
        if titulo or fuente:
            bloque += f"Título: {titulo}\nFuente: {fuente}\n"
        bloque += str(r.get("texto", "")).strip()
        partes_contexto.append(bloque)

    contexto = "\n\n".join(partes_contexto)

    prompt = f"""
Eres un asistente especializado en prevención del blanqueo de capitales y análisis documental.

Responde a la pregunta usando únicamente la información del contexto proporcionado.
No utilices conocimiento externo. Si el contexto no contiene información suficiente, indícalo claramente.
Redacta en español, de forma clara y precisa. No inventes normas, organismos, relaciones ni ejemplos.
Evita respuestas de una sola palabra: da una respuesta suficiente y explicativa, pero sin extenderte innecesariamente.

CONTEXTO:
{contexto}

PREGUNTA:
{pregunta}

RESPUESTA:
"""

    return prompt.strip()


def resumen_prompt(prompt, num_ctx=8192, num_predict=512):
    """Devuelve un diagnóstico aproximado del tamaño del prompt antes de consultar Ollama."""
    caracteres = len(prompt)
    tokens_aprox = round(caracteres / 4)
    margen_aprox = num_ctx - num_predict - tokens_aprox
    return {
        "longitud_prompt_caracteres": caracteres,
        "tokens_prompt_aprox": tokens_aprox,
        "margen_tokens_aprox": margen_aprox
    }


In [18]:
# Probar recuperación y construcción del contexto final con una pregunta

pregunta_prueba = banco[0].get("pregunta", banco[0].get("question", ""))

chunks_prueba = recuperar_chunks(pregunta_prueba, top_k=TOP_K_CHUNKS)
unidades_prueba = recuperar_unidades_grafo(pregunta_prueba, top_k=TOP_K_GRAFO)

activacion_prueba = activar_nodos(
    pregunta=pregunta_prueba,
    chunks_recuperados=chunks_prueba,
    unidades_grafo_recuperadas=unidades_prueba,
    max_nodos_activados=MAX_NODOS_ACTIVADOS
)

nodos_vecinos_prueba, aristas_vecinas_prueba = expandir_por_vecinos(
    activacion_prueba["nodos_activados"],
    profundidad=PROFUNDIDAD_GRAFO,
    max_nodos_vecinos=MAX_NODOS_VECINOS,
    max_aristas=MAX_ARISTAS_VECINAS
)

contexto_prueba = construir_contexto_final(
    chunks_recuperados=chunks_prueba,
    unidades_grafo_recuperadas=unidades_prueba,
    nodos_activados=activacion_prueba["nodos_activados"],
    nodos_vecinos=nodos_vecinos_prueba,
    aristas_vecinas=aristas_vecinas_prueba
)

prompt_prueba = construir_prompt(pregunta_prueba, contexto_prueba)

print("PREGUNTA:")
print(pregunta_prueba)

print("\nChunks recuperados:", len(chunks_prueba))
print("Unidades de grafo recuperadas:", len(unidades_prueba))
print("Entidades localizadas en pregunta:", [e["label"] for e in activacion_prueba["entidades_pregunta"]])
print("Entidades localizadas en chunks:", [e["label"] for e in activacion_prueba["entidades_chunks"][:10]])
print("Nodos activados detectados:", len(activacion_prueba.get("nodos_activados_detectados", [])))
print("Nodos activados usados:", len(activacion_prueba["nodos_activados"]))
print("Nodos vecinos:", len(nodos_vecinos_prueba))
print("Aristas vecinas:", len(aristas_vecinas_prueba))
print("Fragmentos de contexto final:", len(contexto_prueba))
print("Tipos de contexto:", conteo_tipos_contexto(contexto_prueba))
print("Diagnóstico del prompt:", resumen_prompt(prompt_prueba, NUM_CTX_GENERACION, NUM_PREDICT_GENERACION))

for r in contexto_prueba[:8]:
    print("\n" + "="*80)
    print("Rank:", r["rank"], "| tipo:", r.get("metadatos", {}).get("tipo_contexto"))
    print(r["texto"][:700])


PREGUNTA:
¿Qué es el lavado de dinero?

Chunks recuperados: 5
Unidades de grafo recuperadas: 5
Entidades localizadas en pregunta: ['Lavado de dinero']
Entidades localizadas en chunks: ['Casino', 'Delincuencia organizada', 'Economía sumergida', 'Entidades de crédito', 'Extorsión', 'Lavado de dinero', 'Narcotráfico', 'Paraíso fiscal', 'Contrabando', 'Dinero negro']
Nodos activados detectados: 30
Nodos activados usados: 8
Nodos vecinos: 6
Aristas vecinas: 8
Fragmentos de contexto final: 24
Tipos de contexto: Counter({'nodo_activado': 8, 'arista_vecina': 6, 'chunk_faiss': 5, 'unidad_grafo': 5})
Diagnóstico del prompt: {'longitud_prompt_caracteres': 18029, 'tokens_prompt_aprox': 4507, 'margen_tokens_aprox': 3173}

Rank: 1 | tipo: chunk_faiss
El lavado de dinero (también conocido como lavado de capitales, lavado de activos, blanqueo de dinero,​ o blanqueo de capital) es el proceso de cambiar grandes cantidades de dinero obtenidas de delitos, como el narcotráfico, por otras de origen legítimo

In [42]:
# Instalar Ollama

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

^C
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 76 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [43]:
# Iniciar servidor de Ollama

import subprocess
import time

proceso_ollama = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Servidor Ollama iniciado.")

Servidor Ollama iniciado.


In [44]:
# Descargar Llama 3.2 3B

!ollama pull llama3.2:3b

In [45]:
import requests

payload = {
    "model": "llama3.2:3b",
    "prompt": "Responde solo con la palabra: funcionando",
    "stream": False,
    "options": {
        "temperature": 0.0,
        "num_ctx": 2048,
        "num_predict": 20
    }
}

r = requests.post("http://localhost:11434/api/generate", json=payload, timeout=60)
print(r.status_code)
print(r.text[:1000])

200
{"model":"llama3.2:3b","created_at":"2026-06-11T18:21:31.0220083Z","response":"Sí.","done":true,"done_reason":"stop","context":[128006,9125,128007,271,38766,1303,33025,2696,25,6790,220,2366,18,271,128009,128006,882,128007,271,1079,76149,13839,390,1208,74037,25,28963,4988,128009,128006,78191,128007,271,50,2483,13],"total_duration":4955245239,"load_duration":4663250520,"prompt_eval_count":34,"prompt_eval_duration":231358000,"eval_count":4,"eval_duration":57547000}


In [46]:
# Probar Ollama e incluir métricas de eficiencia

import requests
import json


def consultar_ollama(
    prompt,
    modelo="llama3.2:3b",
    temperatura=0.0,
    devolver_metricas=False,
    num_ctx=8192,
    num_predict=512,
    timeout=300,
    formato_json=False
):
    """
    Consulta un modelo local servido por Ollama.

    - devolver_metricas=True permite recoger conteos de tokens y duraciones.
    - num_ctx se fija explícitamente para evitar depender del contexto por defecto.
    - num_predict limita la longitud máxima de la respuesta generada.
    - timeout controla el tiempo máximo de espera de la petición.
    - formato_json=True solicita a Ollama salida JSON, útil para el LLM como juez.
    """

    url = "http://localhost:11434/api/generate"

    payload = {
        "model": modelo,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperatura,
            "num_ctx": num_ctx,
            "num_predict": num_predict
        }
    }

    if formato_json:
        payload["format"] = "json"

    try:
        respuesta = requests.post(url, json=payload, timeout=timeout)
        respuesta.raise_for_status()

    except requests.exceptions.ConnectionError as e:
        raise RuntimeError(
            "No se pudo conectar con Ollama. Comprueba que el servidor esté activo con: "
            "!nohup ollama serve > ollama.log 2>&1 &"
        ) from e

    except requests.exceptions.ReadTimeout as e:
        raise RuntimeError(
            f"Ollama no respondió antes de {timeout} segundos. "
            "Puedes aumentar el timeout o reducir el tamaño del contexto/prompt."
        ) from e

    except requests.exceptions.HTTPError as e:
        raise RuntimeError(
            f"Ollama devolvió un error HTTP: {respuesta.status_code}\n"
            f"Respuesta: {respuesta.text[:1000]}"
        ) from e

    datos = respuesta.json()
    texto_generado = datos.get("response", "")

    metricas = {
        "prompt_eval_count": datos.get("prompt_eval_count"),
        "eval_count": datos.get("eval_count"),
        "total_duration": datos.get("total_duration"),
        "load_duration": datos.get("load_duration"),
        "prompt_eval_duration": datos.get("prompt_eval_duration"),
        "eval_duration": datos.get("eval_duration"),
        "total_duration_s": datos.get("total_duration") / 1e9 if datos.get("total_duration") is not None else None,
        "load_duration_s": datos.get("load_duration") / 1e9 if datos.get("load_duration") is not None else None,
        "prompt_eval_duration_s": datos.get("prompt_eval_duration") / 1e9 if datos.get("prompt_eval_duration") is not None else None,
        "eval_duration_s": datos.get("eval_duration") / 1e9 if datos.get("eval_duration") is not None else None,
    }

    if devolver_metricas:
        return texto_generado, metricas

    return texto_generado


# Prueba mínima de funcionamiento
prueba, metricas_prueba = consultar_ollama(
    "En el contexto de inteligencia artificial, responde en una frase: ¿qué significa RAG?",
    devolver_metricas=True,
    num_ctx=2048,
    num_predict=80,
    timeout=120
)

print("Respuesta generada:")
print(prueba)
print("\nMétricas de eficiencia:")
print(json.dumps(metricas_prueba, ensure_ascii=False, indent=2))


Respuesta generada:
RAG es un acrónimo que se refiere a "Reinforcement Learning with Actor-Critic", una técnica de aprendizaje automático que combina la experiencia y el crítico para mejorar las acciones en un entorno.

Métricas de eficiencia:
{
  "prompt_eval_count": 45,
  "eval_count": 52,
  "total_duration": 1337772319,
  "load_duration": 431243756,
  "prompt_eval_duration": 45159000,
  "eval_duration": 858082000,
  "total_duration_s": 1.337772319,
  "load_duration_s": 0.431243756,
  "prompt_eval_duration_s": 0.045159,
  "eval_duration_s": 0.858082
}


In [47]:
# Ejecutar RAG de grafos sobre todo el banco de evaluación con métricas de eficiencia

from tqdm import tqdm
import time
import json

resultados_rag = []

for item in tqdm(banco):
    pregunta = item.get("pregunta", item.get("question", ""))
    id_pregunta = item.get("id", "")

    inicio_total = time.time()

    try:
        inicio_recuperacion = time.time()

        # 1. Recuperación inicial por chunks, como en el RAG vectorial.
        chunks_recuperados = recuperar_chunks(pregunta, top_k=TOP_K_CHUNKS)

        # 2. Recuperación semántica sobre unidades del grafo.
        unidades_grafo_recuperadas = recuperar_unidades_grafo(pregunta, top_k=TOP_K_GRAFO)

        # 3. Activación de entidades por pregunta, chunks y unidades del grafo.
        activacion = activar_nodos(
            pregunta=pregunta,
            chunks_recuperados=chunks_recuperados,
            unidades_grafo_recuperadas=unidades_grafo_recuperadas,
            max_nodos_activados=MAX_NODOS_ACTIVADOS
        )

        nodos_activados_usados = activacion["nodos_activados"]

        # 4. Expansión por vecinos del grafo.
        nodos_vecinos, aristas_vecinas = expandir_por_vecinos(
            nodos_activados_usados,
            profundidad=PROFUNDIDAD_GRAFO,
            max_nodos_vecinos=MAX_NODOS_VECINOS,
            max_aristas=MAX_ARISTAS_VECINAS
        )

        # 5. Construcción compacta del contexto final.
        contexto_final = construir_contexto_final(
            chunks_recuperados=chunks_recuperados,
            unidades_grafo_recuperadas=unidades_grafo_recuperadas,
            nodos_activados=nodos_activados_usados,
            nodos_vecinos=nodos_vecinos,
            aristas_vecinas=aristas_vecinas,
            max_unidades_grafo=MAX_UNIDADES_GRAFO,
            max_nodos_activados=MAX_NODOS_ACTIVADOS,
            max_nodos_vecinos=MAX_NODOS_VECINOS,
            max_aristas_vecinas=MAX_ARISTAS_VECINAS,
            presupuesto_contexto=MAX_CARACTERES_CONTEXTO_GENERACION
        )

        fin_recuperacion = time.time()

        prompt = construir_prompt(pregunta, contexto_final)

        # Control preventivo: si el prompt queda demasiado largo, se recorta el contexto final.
        recorte_preventivo = False
        if len(prompt) > MAX_LONGITUD_PROMPT_GENERACION:
            recorte_preventivo = True
            contexto_final = recortar_contexto_final_por_presupuesto(
                contexto_final,
                max_caracteres=MAX_CARACTERES_CONTEXTO_REINTENTO
            )
            prompt = construir_prompt(pregunta, contexto_final)

        diagnostico_prompt = resumen_prompt(
            prompt,
            num_ctx=NUM_CTX_GENERACION,
            num_predict=NUM_PREDICT_GENERACION
        )

        inicio_generacion = time.time()
        respuesta_generada, metricas_ollama = consultar_ollama(
            prompt,
            modelo=MODELO_GENERATIVO,
            temperatura=0.0,
            devolver_metricas=True,
            num_ctx=NUM_CTX_GENERACION,
            num_predict=NUM_PREDICT_GENERACION,
            timeout=TIMEOUT_GENERACION
        )
        fin_generacion = time.time()

        # Control posterior: si el modelo quedó sin espacio de salida, se reintenta con contexto más corto.
        tokens_entrada = metricas_ollama.get("prompt_eval_count")
        tokens_salida = metricas_ollama.get("eval_count")
        respuesta_muy_corta = len(str(respuesta_generada).strip()) < 25 and (tokens_salida is not None and tokens_salida <= 5)
        prompt_casi_lleno = tokens_entrada is not None and tokens_entrada >= (NUM_CTX_GENERACION - 128)
        reintento_por_contexto = False

        if respuesta_muy_corta or prompt_casi_lleno:
            reintento_por_contexto = True
            contexto_final = recortar_contexto_final_por_presupuesto(
                contexto_final,
                max_caracteres=MAX_CARACTERES_CONTEXTO_REINTENTO
            )
            prompt = construir_prompt(pregunta, contexto_final)
            diagnostico_prompt = resumen_prompt(
                prompt,
                num_ctx=NUM_CTX_GENERACION,
                num_predict=NUM_PREDICT_GENERACION
            )

            inicio_generacion = time.time()
            respuesta_generada, metricas_ollama = consultar_ollama(
                prompt,
                modelo=MODELO_GENERATIVO,
                temperatura=0.0,
                devolver_metricas=True,
                num_ctx=NUM_CTX_GENERACION,
                num_predict=NUM_PREDICT_GENERACION,
                timeout=TIMEOUT_GENERACION
            )
            fin_generacion = time.time()

        fin_total = time.time()

        resultado = {
            "id": id_pregunta,
            "pregunta": pregunta,
            "datos_banco": item,

            "top_k_chunks": TOP_K_CHUNKS,
            "top_k_grafo": TOP_K_GRAFO,
            "profundidad_grafo": PROFUNDIDAD_GRAFO,
            "max_unidades_grafo": MAX_UNIDADES_GRAFO,
            "max_nodos_activados": MAX_NODOS_ACTIVADOS,
            "max_nodos_vecinos": MAX_NODOS_VECINOS,
            "max_aristas_vecinas": MAX_ARISTAS_VECINAS,
            "max_caracteres_contexto_generacion": MAX_CARACTERES_CONTEXTO_GENERACION,
            "max_longitud_prompt_generacion": MAX_LONGITUD_PROMPT_GENERACION,

            "modelo_embeddings": MODELO_EMBEDDINGS,
            "modelo_generativo": MODELO_GENERATIVO,
            "num_ctx_generacion": NUM_CTX_GENERACION,
            "num_predict_generacion": NUM_PREDICT_GENERACION,
            "metadata_grafo": metadata_grafo,

            "chunks_recuperados": chunks_recuperados,
            "unidades_grafo_recuperadas": unidades_grafo_recuperadas,
            "entidades_pregunta": activacion["entidades_pregunta"],
            "entidades_chunks": activacion["entidades_chunks"],
            "nodos_activados_detectados": activacion.get("nodos_activados_detectados", []),
            "nodos_activados_usados": nodos_activados_usados,
            "nodos_vecinos_usados": nodos_vecinos,
            "aristas_vecinas_usadas": aristas_vecinas,
            "contexto_final_recuperado": contexto_final,
            "tipos_contexto_final": dict(conteo_tipos_contexto(contexto_final)),

            "respuesta_generada": respuesta_generada,

            # Latencia total del sistema RAG: recuperación/expansión + construcción del prompt + generación.
            "latencia_segundos": fin_total - inicio_total,
            "latencia_recuperacion_segundos": fin_recuperacion - inicio_recuperacion,
            "latencia_generacion_segundos": fin_generacion - inicio_generacion,

            # Métricas devueltas por Ollama.
            "tokens_entrada": metricas_ollama.get("prompt_eval_count"),
            "tokens_salida": metricas_ollama.get("eval_count"),
            "duracion_total_s": metricas_ollama.get("total_duration_s"),
            "duracion_carga_s": metricas_ollama.get("load_duration_s"),
            "duracion_prompt_s": metricas_ollama.get("prompt_eval_duration_s"),
            "duracion_generacion_s": metricas_ollama.get("eval_duration_s"),

            # Control adicional para detectar prompts excesivos.
            "longitud_prompt_caracteres": len(prompt),
            "tokens_prompt_aprox": diagnostico_prompt.get("tokens_prompt_aprox"),
            "margen_tokens_aprox": diagnostico_prompt.get("margen_tokens_aprox"),
            "recorte_preventivo_prompt": recorte_preventivo,
            "reintento_por_contexto": reintento_por_contexto,
            "respuesta_posiblemente_cortada": (
                metricas_ollama.get("eval_count") is not None
                and metricas_ollama.get("eval_count") <= 5
                and len(str(respuesta_generada).strip()) < 25
            ),

            # Estado de la ejecución.
            "estado": "ok",
            "error": None
        }

    except Exception as e:
        fin_total = time.time()

        resultado = {
            "id": id_pregunta,
            "pregunta": pregunta,
            "datos_banco": item,
            "estado": "error",
            "error": str(e),
            "latencia_segundos": fin_total - inicio_total
        }

        print(f"\nError en {id_pregunta}: {e}")

    resultados_rag.append(resultado)

    # Guardado parcial después de cada pregunta.
    with open(RUTA_RESULTADOS_RAG_GRAFOS_PARCIAL, "w", encoding="utf-8") as f:
        json.dump(resultados_rag, f, ensure_ascii=False, indent=2)

    # Aviso visible si algo sigue quedando raro.
    if resultado.get("estado") == "ok":
        if resultado.get("respuesta_posiblemente_cortada"):
            print(f"\nAVISO: respuesta posiblemente cortada en {id_pregunta}")
        if resultado.get("tokens_entrada") and resultado["tokens_entrada"] > (NUM_CTX_GENERACION - NUM_PREDICT_GENERACION):
            print(f"\nAVISO: prompt muy largo en {id_pregunta}: {resultado['tokens_entrada']} tokens de entrada")

print("Preguntas procesadas:", len(resultados_rag))
print("Resultados guardados en:", RUTA_RESULTADOS_RAG_GRAFOS_PARCIAL)


100%|██████████| 30/30 [04:42<00:00,  9.43s/it]

Preguntas procesadas: 30
Resultados guardados en: /content/resultados_rag_grafos_parcial.json


In [48]:
# Guardar resultados en JSON base

with open(RUTA_SALIDA_JSON, "w", encoding="utf-8") as f:
    json.dump(resultados_rag, f, ensure_ascii=False, indent=2)

print("Archivo JSON guardado en:")
print(RUTA_SALIDA_JSON)

Archivo JSON guardado en:
/content/resultados_rag_grafos.json


In [49]:
# Funciones auxiliares para la evaluación automática mediante LLM como juez
# Inspirada en RAGAS, pero integrada en un único procedimiento de evaluación.

import re
import json

def extraer_json(texto):
    """
    Extrae un objeto JSON de la salida del modelo evaluador.
    La función contempla el caso ideal, en el que el modelo devuelve únicamente JSON,
    y el caso frecuente en el que añade texto antes o después.
    """
    try:
        return json.loads(texto)
    except Exception:
        pass

    patron = r"\{[\s\S]*\}"
    coincidencia = re.search(patron, texto)

    if coincidencia:
        try:
            return json.loads(coincidencia.group(0))
        except Exception:
            return None

    return None


def limitar_texto(texto, max_caracteres=1800):
    """
    Limita la longitud de cada fragmento para evitar prompts de evaluación demasiado extensos.
    """
    if texto is None:
        return ""

    texto = str(texto)

    if len(texto) <= max_caracteres:
        return texto

    return texto[:max_caracteres] + "\n[Texto truncado]"


def normalizar_puntuacion(valor):
    """
    Convierte una puntuación a float, la limita al intervalo 0.0-1.0
    y la redondea a un decimal, de acuerdo con la escala definida en la memoria.
    """
    try:
        valor = float(valor)
    except Exception:
        return None

    valor = max(0.0, min(1.0, valor))
    return round(valor, 1)


def normalizar_evaluacion_llm(datos, salida_original=None):
    """
    Normaliza la salida del evaluador para asegurar que todas las dimensiones
    estén presentes con los nombres empleados en el diseño de evaluación.
    """
    if not isinstance(datos, dict):
        return {
            "relevancia_contexto": None,
            "relevancia_respuesta": None,
            "fidelidad_respuesta_contexto": None,
            "justificacion_relevancia_contexto": "",
            "justificacion_relevancia_respuesta": "",
            "justificacion_fidelidad_respuesta_contexto": "",
            "justificacion_breve": "No se pudo extraer un JSON válido.",
            "salida_original": salida_original
        }

    evaluacion = {
        "relevancia_contexto": normalizar_puntuacion(datos.get("relevancia_contexto")),
        "relevancia_respuesta": normalizar_puntuacion(datos.get("relevancia_respuesta")),
        "fidelidad_respuesta_contexto": normalizar_puntuacion(datos.get("fidelidad_respuesta_contexto")),
        "justificacion_relevancia_contexto": str(datos.get("justificacion_relevancia_contexto", "")),
        "justificacion_relevancia_respuesta": str(datos.get("justificacion_relevancia_respuesta", "")),
        "justificacion_fidelidad_respuesta_contexto": str(datos.get("justificacion_fidelidad_respuesta_contexto", "")),
        "justificacion_breve": str(datos.get("justificacion_breve", ""))
    }

    puntuaciones = [
        evaluacion["relevancia_contexto"],
        evaluacion["relevancia_respuesta"],
        evaluacion["fidelidad_respuesta_contexto"]
    ]

    puntuaciones_validas = [p for p in puntuaciones if p is not None]

    if puntuaciones_validas:
        evaluacion["media_evaluacion_automatica"] = round(
            sum(puntuaciones_validas) / len(puntuaciones_validas),
            3
        )
    else:
        evaluacion["media_evaluacion_automatica"] = None

    return evaluacion

In [50]:
# Prompt y función de evaluación automática mediante LLM como juez
# Se mantiene el mismo tipo de evaluación que en el RAG vectorial.
# El contexto evaluado es el contexto final del RAG de grafos.
# Diseño: role-play + few-shot + escala explícita.

NUM_CTX_EVALUADOR = 8192
NUM_PREDICT_EVALUADOR = 512
TIMEOUT_EVALUADOR = 300
MAX_FRAGMENTOS_EVALUADOR = 12
MAX_CARACTERES_FRAGMENTO_EVALUADOR = 700
MAX_CARACTERES_CONTEXTO_EVALUADOR = 9000


def construir_contexto_evaluador(chunks_recuperados):
    contexto = ""
    usados = 0

    for r in chunks_recuperados[:MAX_FRAGMENTOS_EVALUADOR]:
        metadatos = r.get("metadatos", {})
        doc_id = metadatos.get("doc_id", "")
        chunk_id = metadatos.get("chunk_id", "")
        titulo = metadatos.get("titulo", "")
        tipo_contexto = metadatos.get("tipo_contexto", "")

        texto_chunk = limitar_texto(
            r.get("texto", ""),
            max_caracteres=MAX_CARACTERES_FRAGMENTO_EVALUADOR
        )

        bloque = (
            f"\n[Fragmento {r.get('rank', '')} | tipo={tipo_contexto} | "
            f"doc_id={doc_id} | chunk_id={chunk_id} | título={titulo}]\n"
            f"{texto_chunk}\n"
        )

        if usados + len(bloque) > MAX_CARACTERES_CONTEXTO_EVALUADOR:
            restante = MAX_CARACTERES_CONTEXTO_EVALUADOR - usados
            if restante > 300:
                contexto += limitar_texto(bloque, max_caracteres=restante)
            break

        contexto += bloque
        usados += len(bloque)

    return contexto


def construir_prompt_evaluador_llm(pregunta, respuesta, chunks_recuperados):
    contexto = construir_contexto_evaluador(chunks_recuperados)

    prompt_eval = f"""
Actúa como evaluador externo experto en sistemas RAG aplicados a un dominio especializado.
Evalúa una respuesta generada por un sistema RAG a partir de una pregunta y de un contexto recuperado.

La evaluación está inspirada en RAGAS, pero se formula como una única evaluación automática mediante LLM como juez.
Valora tres dimensiones con puntuaciones de 0.0 a 1.0 y un decimal:

1. relevancia_contexto: utilidad del contexto para responder a la pregunta.
0.0 irrelevante; 0.1-0.3 muy poco útil; 0.4-0.6 parcialmente útil; 0.7-0.8 bastante útil; 0.9-1.0 muy pertinente.

2. relevancia_respuesta: adecuación de la respuesta a la pregunta.
0.0 no responde o es incorrecta; 0.1-0.3 muy deficiente; 0.4-0.6 parcial o incompleta; 0.7-0.8 buena con pequeños fallos; 0.9-1.0 muy buena.

3. fidelidad_respuesta_contexto: apoyo de las afirmaciones de la respuesta en el contexto.
0.0 no apoyada o inventada; 0.1-0.3 muy poco apoyada; 0.4-0.6 parcialmente apoyada; 0.7-0.8 mayoritariamente apoyada; 0.9-1.0 plenamente apoyada.

Instrucciones:
- Analiza internamente el caso antes de puntuar, pero no muestres razonamiento paso a paso.
- Distingue entre relevancia de la respuesta y fidelidad al contexto.
- Una respuesta puede ser fiel pero incompleta.
- Un contexto puede ser útil aunque la respuesta sea mala.
- Devuelve únicamente JSON válido. No añadas texto fuera del JSON.

Ejemplo:
PREGUNTA: ¿Qué es la diligencia debida del cliente?
CONTEXTO: La diligencia debida del cliente comprende la identificación del cliente, la verificación de su identidad y el seguimiento continuo de la relación de negocios.
RESPUESTA: La diligencia debida del cliente consiste en identificar al cliente, verificar su identidad y realizar un seguimiento continuo.
JSON:
{{
  "relevancia_contexto": 1.0,
  "relevancia_respuesta": 0.9,
  "fidelidad_respuesta_contexto": 1.0,
  "justificacion_relevancia_contexto": "El contexto contiene directamente la información necesaria.",
  "justificacion_relevancia_respuesta": "La respuesta responde de forma correcta y suficiente.",
  "justificacion_fidelidad_respuesta_contexto": "Las afirmaciones están apoyadas por el contexto.",
  "justificacion_breve": "Contexto útil, respuesta adecuada y fiel."
}}

Ahora evalúa el siguiente caso real.

PREGUNTA:
{pregunta}

CONTEXTO RECUPERADO:
{contexto}

RESPUESTA GENERADA:
{respuesta}

JSON:
"""

    return prompt_eval


def evaluar_llm_como_juez(pregunta, respuesta, chunks_recuperados, modelo_evaluador=None):
    """
    Evalúa automáticamente un caso mediante un LLM como juez.
    El modelo recibe la pregunta, el contexto recuperado y la respuesta generada.
    """
    if modelo_evaluador is None:
        modelo_evaluador = MODELO_GENERATIVO

    prompt_eval = construir_prompt_evaluador_llm(
        pregunta=pregunta,
        respuesta=respuesta,
        chunks_recuperados=chunks_recuperados
    )

    salida = consultar_ollama(
        prompt_eval,
        modelo=modelo_evaluador,
        temperatura=0.0,
        num_ctx=NUM_CTX_EVALUADOR,
        num_predict=NUM_PREDICT_EVALUADOR,
        timeout=TIMEOUT_EVALUADOR,
        formato_json=True
    )

    datos = extraer_json(salida)

    return normalizar_evaluacion_llm(datos, salida_original=salida)


In [51]:
# Ejecutar evaluación automática mediante LLM como juez inspirada en RAGAS

MODELO_EVALUADOR = MODELO_GENERATIVO

for r in tqdm(resultados_rag):
    if r.get("estado") != "ok":
        r["modelo_evaluador"] = MODELO_EVALUADOR
        r["num_ctx_evaluador"] = NUM_CTX_EVALUADOR
        r["eval_relevancia_contexto"] = None
        r["eval_relevancia_respuesta"] = None
        r["eval_fidelidad_respuesta_contexto"] = None
        r["eval_media_automatica"] = None
        r["eval_justificacion_breve"] = "No se evalúa automáticamente porque la generación terminó con error."
        continue

    evaluacion = evaluar_llm_como_juez(
        pregunta=r["pregunta"],
        respuesta=r["respuesta_generada"],
        chunks_recuperados=r["contexto_final_recuperado"],
        modelo_evaluador=MODELO_EVALUADOR
    )

    r["modelo_evaluador"] = MODELO_EVALUADOR
    r["num_ctx_evaluador"] = NUM_CTX_EVALUADOR

    r["eval_relevancia_contexto"] = evaluacion.get("relevancia_contexto")
    r["eval_relevancia_respuesta"] = evaluacion.get("relevancia_respuesta")
    r["eval_fidelidad_respuesta_contexto"] = evaluacion.get("fidelidad_respuesta_contexto")
    r["eval_media_automatica"] = evaluacion.get("media_evaluacion_automatica")

    r["eval_justificacion_relevancia_contexto"] = evaluacion.get("justificacion_relevancia_contexto", "")
    r["eval_justificacion_relevancia_respuesta"] = evaluacion.get("justificacion_relevancia_respuesta", "")
    r["eval_justificacion_fidelidad_respuesta_contexto"] = evaluacion.get("justificacion_fidelidad_respuesta_contexto", "")
    r["eval_justificacion_breve"] = evaluacion.get("justificacion_breve", "")

    if "salida_original" in evaluacion:
        r["eval_salida_original"] = evaluacion["salida_original"]

    # Guardado parcial de la evaluación automática.
    with open(RUTA_SALIDA_JSON_METRICAS if "RUTA_SALIDA_JSON_METRICAS" in globals() else RUTA_SALIDA_JSON, "w", encoding="utf-8") as f:
        json.dump(resultados_rag, f, ensure_ascii=False, indent=2)

print("Evaluación automática mediante LLM como juez completada.")


100%|██████████| 30/30 [01:33<00:00,  3.13s/it]

Evaluación automática mediante LLM como juez completada.


In [52]:
# Guardar JSON final con métricas y evaluación automática

RUTA_SALIDA_JSON_METRICAS = os.path.join(
    RUTA_BASE,
    "resultados_rag_grafos_con_evaluacion_automatica.json"
)

with open(RUTA_SALIDA_JSON_METRICAS, "w", encoding="utf-8") as f:
    json.dump(resultados_rag, f, ensure_ascii=False, indent=2)

print("JSON con métricas y evaluación automática guardado en:")
print(RUTA_SALIDA_JSON_METRICAS)

JSON con métricas y evaluación automática guardado en:
/content/resultados_rag_grafos_con_evaluacion_automatica.json


In [53]:
# Crear CSV final con métricas de eficiencia, recuperación de grafo y evaluación automática

import pandas as pd

filas = []

for r in resultados_rag:
    fila = {
        "id": r.get("id"),
        "pregunta": r.get("pregunta"),
        "respuesta_generada": r.get("respuesta_generada"),
        "estado": r.get("estado"),
        "error": r.get("error"),

        "top_k_chunks": r.get("top_k_chunks"),
        "top_k_grafo": r.get("top_k_grafo"),
        "profundidad_grafo": r.get("profundidad_grafo"),
        "max_unidades_grafo": r.get("max_unidades_grafo"),
        "max_nodos_activados": r.get("max_nodos_activados"),
        "max_nodos_vecinos": r.get("max_nodos_vecinos"),
        "max_aristas_vecinas": r.get("max_aristas_vecinas"),

        "modelo_embeddings": r.get("modelo_embeddings"),
        "modelo_generativo": r.get("modelo_generativo"),
        "modelo_evaluador": r.get("modelo_evaluador"),
        "num_ctx_generacion": r.get("num_ctx_generacion"),
        "num_predict_generacion": r.get("num_predict_generacion"),
        "num_ctx_evaluador": r.get("num_ctx_evaluador"),

        "num_chunks_recuperados": len(r.get("chunks_recuperados", [])),
        "num_unidades_grafo_recuperadas": len(r.get("unidades_grafo_recuperadas", [])),
        "num_entidades_pregunta": len(r.get("entidades_pregunta", [])),
        "num_entidades_chunks": len(r.get("entidades_chunks", [])),
        "num_nodos_activados_detectados": len(r.get("nodos_activados_detectados", [])),
        "num_nodos_activados_usados": len(r.get("nodos_activados_usados", [])),
        "num_nodos_vecinos_usados": len(r.get("nodos_vecinos_usados", [])),
        "num_aristas_vecinas_usadas": len(r.get("aristas_vecinas_usadas", [])),
        "num_fragmentos_contexto_final": len(r.get("contexto_final_recuperado", [])),

        "tipos_contexto_final": json.dumps(r.get("tipos_contexto_final", {}), ensure_ascii=False),
        "entidades_pregunta_labels": "; ".join(e.get("label", "") for e in r.get("entidades_pregunta", [])),
        "entidades_chunks_labels": "; ".join(e.get("label", "") for e in r.get("entidades_chunks", [])),
        "nodos_activados_detectados": "; ".join(r.get("nodos_activados_detectados", [])),
        "nodos_activados_usados": "; ".join(r.get("nodos_activados_usados", [])),
        "nodos_vecinos_usados": "; ".join(r.get("nodos_vecinos_usados", [])),

        "latencia_segundos": r.get("latencia_segundos"),
        "latencia_recuperacion_segundos": r.get("latencia_recuperacion_segundos"),
        "latencia_generacion_segundos": r.get("latencia_generacion_segundos"),
        "tokens_entrada": r.get("tokens_entrada"),
        "tokens_salida": r.get("tokens_salida"),
        "longitud_prompt_caracteres": r.get("longitud_prompt_caracteres"),
        "tokens_prompt_aprox": r.get("tokens_prompt_aprox"),
        "margen_tokens_aprox": r.get("margen_tokens_aprox"),
        "recorte_preventivo_prompt": r.get("recorte_preventivo_prompt"),
        "reintento_por_contexto": r.get("reintento_por_contexto"),
        "respuesta_posiblemente_cortada": r.get("respuesta_posiblemente_cortada"),

        "duracion_total_s": r.get("duracion_total_s"),
        "duracion_carga_s": r.get("duracion_carga_s"),
        "duracion_prompt_s": r.get("duracion_prompt_s"),
        "duracion_generacion_s": r.get("duracion_generacion_s"),

        "eval_relevancia_contexto": r.get("eval_relevancia_contexto"),
        "eval_relevancia_respuesta": r.get("eval_relevancia_respuesta"),
        "eval_fidelidad_respuesta_contexto": r.get("eval_fidelidad_respuesta_contexto"),
        "eval_media_automatica": r.get("eval_media_automatica"),

        "eval_justificacion_relevancia_contexto": r.get("eval_justificacion_relevancia_contexto"),
        "eval_justificacion_relevancia_respuesta": r.get("eval_justificacion_relevancia_respuesta"),
        "eval_justificacion_fidelidad_respuesta_contexto": r.get("eval_justificacion_fidelidad_respuesta_contexto"),
        "eval_justificacion_breve": r.get("eval_justificacion_breve")
    }

    datos_banco = r.get("datos_banco", {})

    for clave, valor in datos_banco.items():
        if clave not in fila:
            if isinstance(valor, (dict, list)):
                fila[f"banco_{clave}"] = json.dumps(valor, ensure_ascii=False)
            else:
                fila[f"banco_{clave}"] = valor

    for chunk in r.get("chunks_recuperados", []):
        n = chunk.get("rank")
        metadatos = chunk.get("metadatos", {})
        fila[f"chunk_{n}_score"] = chunk.get("score")
        fila[f"chunk_{n}_texto"] = chunk.get("texto")
        fila[f"chunk_{n}_indice"] = chunk.get("indice_chunk")
        fila[f"chunk_{n}_doc_id"] = metadatos.get("doc_id", "")
        fila[f"chunk_{n}_chunk_id"] = metadatos.get("chunk_id", "")
        fila[f"chunk_{n}_titulo"] = metadatos.get("titulo", "")
        fila[f"chunk_{n}_fuente"] = metadatos.get("fuente", "")

    for unidad in r.get("unidades_grafo_recuperadas", [])[:10]:
        n = unidad.get("rank")
        fila[f"grafo_unidad_{n}_score"] = unidad.get("score")
        fila[f"grafo_unidad_{n}_tipo"] = unidad.get("tipo_unidad")
        fila[f"grafo_unidad_{n}_id"] = unidad.get("id_unidad")
        fila[f"grafo_unidad_{n}_node_id"] = unidad.get("node_id", "")
        fila[f"grafo_unidad_{n}_edge_id"] = unidad.get("edge_id", "")
        fila[f"grafo_unidad_{n}_texto"] = unidad.get("texto", "")

    filas.append(fila)

df_resultados_metricas = pd.DataFrame(filas)

RUTA_SALIDA_CSV_METRICAS = os.path.join(
    RUTA_BASE,
    "resultados_rag_grafos_con_evaluacion_automatica.csv"
)

df_resultados_metricas.to_csv(
    RUTA_SALIDA_CSV_METRICAS,
    index=False,
    encoding="utf-8-sig"
)

print("CSV con métricas y evaluación automática guardado en:")
print(RUTA_SALIDA_CSV_METRICAS)

df_resultados_metricas.head()


CSV con métricas y evaluación automática guardado en:
/content/resultados_rag_grafos_con_evaluacion_automatica.csv


,id,pregunta,respuesta_generada,estado,error,top_k_chunks,top_k_grafo,profundidad_grafo,max_unidades_grafo,max_nodos_activados,...,grafo_unidad_4_id,grafo_unidad_4_node_id,grafo_unidad_4_edge_id,grafo_unidad_4_texto,grafo_unidad_5_score,grafo_unidad_5_tipo,grafo_unidad_5_id,grafo_unidad_5_node_id,grafo_unidad_5_edge_id,grafo_unidad_5_texto
0,Q001,¿Qué es el lavado de dinero?,El lavado de dinero es el proceso de cambiar g...,ok,None,5,5,1,5,8,...,node::lavado_de_dinero,lavado_de_dinero,None,Nodo del grafo AML/CFT. Entidad: Lavado de din...,0.869024,arista,edge::lavado_de_dinero_incluye_blanqueo_basado...,None,lavado_de_dinero_incluye_blanqueo_basado_en_co...,Relación del grafo AML/CFT. Lavado de dinero -...
1,Q002,¿Qué es el pitufeo?,El pitufeo es una práctica de lavado de activo...,ok,None,5,5,1,5,8,...,evidence::edge::lavado_de_dinero_incluye_pituf...,None,lavado_de_dinero_incluye_pitufeo,Evidencia asociada a la relación Lavado de din...,0.823373,evidencia_arista,evidence::edge::empresa_fantasma_oculta_titula...,None,empresa_fantasma_oculta_titular_real,Evidencia asociada a la relación Empresa fanta...
2,Q003,¿Qué significa persona expuesta políticamente?,Una persona expuesta políticamente (PEP) es un...,ok,None,5,5,1,5,8,...,node::pep,pep,None,Nodo del grafo AML/CFT. Entidad: Persona expue...,0.854725,arista,edge::pep_aumenta_riesgo_lavado_de_dinero,None,pep_aumenta_riesgo_lavado_de_dinero,Relación del grafo AML/CFT. Persona expuesta p...
3,Q004,¿Qué es la diligencia debida?,La diligencia debida es un proceso de investig...,ok,None,5,5,1,5,8,...,edge::diligencia_debida_simplificada_mitiga_la...,None,diligencia_debida_simplificada_mitiga_lavado_d...,Relación del grafo AML/CFT. Diligencia debida ...,0.848592,evidencia_nodo,evidence::node::diligencia_debida_simplificada::1,diligencia_debida_simplificada,None,Evidencia asociada al nodo Diligencia debida s...
4,Q005,La entidad Amroggang Development Banking Corpo...,Para determinar si la entidad Amroggang Develo...,ok,None,5,5,1,5,8,...,node::aplicacion_listas_sanciones,aplicacion_listas_sanciones,None,Nodo del grafo AML/CFT. Entidad: Aplicación de...,0.826832,arista,edge::sanciones_financieras_internacionales_se...,None,sanciones_financieras_internacionales_se_aplic...,Relación del grafo AML/CFT. Sanciones financie...


In [54]:
# Resumen de métricas principales

metricas_resumen = [
    "latencia_segundos",
    "latencia_recuperacion_segundos",
    "latencia_generacion_segundos",
    "tokens_entrada",
    "tokens_salida",
    "longitud_prompt_caracteres",
    "tokens_prompt_aprox",
    "num_chunks_recuperados",
    "num_unidades_grafo_recuperadas",
    "num_entidades_pregunta",
    "num_entidades_chunks",
    "num_nodos_activados_detectados",
    "num_nodos_activados_usados",
    "num_nodos_vecinos_usados",
    "num_aristas_vecinas_usadas",
    "num_fragmentos_contexto_final",
    "eval_relevancia_contexto",
    "eval_relevancia_respuesta",
    "eval_fidelidad_respuesta_contexto",
    "eval_media_automatica"
]

columnas_existentes = [c for c in metricas_resumen if c in df_resultados_metricas.columns]
resumen_metricas = df_resultados_metricas[columnas_existentes].mean(numeric_only=True)

print("Resumen medio de métricas:")
print(resumen_metricas)

print("\nCasos marcados como posiblemente cortados:")
if "respuesta_posiblemente_cortada" in df_resultados_metricas.columns:
    print(df_resultados_metricas["respuesta_posiblemente_cortada"].value_counts(dropna=False))


Resumen medio de métricas:
latencia_segundos                        9.385120
latencia_recuperacion_segundos           0.171648
latencia_generacion_segundos             9.213377
tokens_entrada                        5214.100000
tokens_salida                          306.000000
longitud_prompt_caracteres           18361.633333
tokens_prompt_aprox                   4590.466667
num_chunks_recuperados                   5.000000
num_unidades_grafo_recuperadas           5.000000
num_entidades_pregunta                   1.366667
num_entidades_chunks                    21.966667
num_nodos_activados_detectados          23.366667
num_nodos_activados_usados               8.000000
num_nodos_vecinos_usados                 5.900000
num_aristas_vecinas_usadas               8.433333
num_fragmentos_contexto_final           23.200000
eval_relevancia_contexto                 0.883333
eval_relevancia_respuesta                0.753333
eval_fidelidad_respuesta_contexto        0.713333
eval_media_automatica  

In [55]:
# Crear una tabla resumida con únicamente las métricas principales

columnas_resumen = [
    "id",
    "pregunta",
    "estado",
    "latencia_segundos",
    "latencia_recuperacion_segundos",
    "latencia_generacion_segundos",
    "tokens_entrada",
    "tokens_salida",
    "longitud_prompt_caracteres",
    "tokens_prompt_aprox",
    "recorte_preventivo_prompt",
    "reintento_por_contexto",
    "respuesta_posiblemente_cortada",
    "num_unidades_grafo_recuperadas",
    "num_entidades_pregunta",
    "num_entidades_chunks",
    "num_nodos_activados_detectados",
    "num_nodos_activados_usados",
    "num_nodos_vecinos_usados",
    "num_aristas_vecinas_usadas",
    "num_fragmentos_contexto_final",
    "eval_relevancia_contexto",
    "eval_relevancia_respuesta",
    "eval_fidelidad_respuesta_contexto",
    "eval_media_automatica"
]

columnas_resumen_existentes = [c for c in columnas_resumen if c in df_resultados_metricas.columns]
df_metricas_resumidas = df_resultados_metricas[columnas_resumen_existentes].copy()

# Redondear las métricas para facilitar la lectura
columnas_numericas = df_metricas_resumidas.select_dtypes(include="number").columns
for col in columnas_numericas:
    df_metricas_resumidas[col] = df_metricas_resumidas[col].round(3)

RUTA_SALIDA_CSV_RESUMIDO = os.path.join(
    RUTA_BASE,
    "tabla_resumida_metricas_rag_grafos.csv"
)

df_metricas_resumidas.to_csv(
    RUTA_SALIDA_CSV_RESUMIDO,
    index=False,
    encoding="utf-8-sig"
)

print("Tabla resumida guardada en:")
print(RUTA_SALIDA_CSV_RESUMIDO)

df_metricas_resumidas.head(10)


Tabla resumida guardada en:
/content/tabla_resumida_metricas_rag_grafos.csv


,id,pregunta,estado,latencia_segundos,latencia_recuperacion_segundos,latencia_generacion_segundos,tokens_entrada,tokens_salida,longitud_prompt_caracteres,tokens_prompt_aprox,...,num_entidades_chunks,num_nodos_activados_detectados,num_nodos_activados_usados,num_nodos_vecinos_usados,num_aristas_vecinas_usadas,num_fragmentos_contexto_final,eval_relevancia_contexto,eval_relevancia_respuesta,eval_fidelidad_respuesta_contexto,eval_media_automatica
0,Q001,¿Qué es el lavado de dinero?,ok,10.545,0.149,10.397,5090,119,18029,4507,...,30,30,8,6,8,24,0.9,0.8,0.7,0.800
1,Q002,¿Qué es el pitufeo?,ok,6.258,0.383,5.875,5097,100,18008,4502,...,26,26,8,6,8,24,0.9,0.8,0.7,0.800
2,Q003,¿Qué significa persona expuesta políticamente?,ok,7.545,0.176,7.369,5078,67,18488,4622,...,28,28,8,6,10,24,1.0,0.9,0.8,0.900
3,Q004,¿Qué es la diligencia debida?,ok,5.566,0.119,5.447,5392,94,18743,4686,...,17,17,8,6,8,24,0.9,0.8,0.7,0.800
4,Q005,La entidad Amroggang Development Banking Corpo...,ok,9.038,0.141,8.897,5436,267,19413,4853,...,27,30,8,6,10,24,0.8,0.2,0.6,0.533
5,Q006,¿Cuál es la diferencia entre evasión fiscal y ...,ok,8.883,0.217,8.666,5209,279,18332,4583,...,15,15,8,6,10,25,0.9,0.8,0.7,0.800
6,Q007,¿En qué se diferencian una empresa fantasma y ...,ok,8.875,0.154,8.722,4932,330,18034,4508,...,19,20,8,6,9,24,0.9,0.8,0.9,0.867
7,Q008,¿Qué diferencia existe entre dinero negro y ec...,ok,6.599,0.139,6.460,4974,197,18041,4510,...,25,25,8,5,10,24,0.9,0.8,0.7,0.800
8,Q009,¿Qué relación existe entre las personas expues...,ok,8.303,0.107,8.196,5567,277,18870,4718,...,23,25,8,5,10,24,0.9,0.8,0.7,0.800
9,Q010,¿Cómo se relaciona la propiedad efectiva con e...,ok,9.829,0.140,9.688,5069,374,18659,4665,...,14,14,8,6,9,24,0.8,0.7,0.9,0.800


In [56]:
# Crear un archivo sencillo para la evaluación de la experta
# Incluye identificador, pregunta, respuesta generada, puntuación cuantitativa y comentarios.

import pandas as pd
import os

filas_experta = []

for r in resultados_rag:
    datos_banco = r.get("datos_banco", {})
    filas_experta.append({
        "id": r.get("id"),
        "pregunta": r.get("pregunta"),
        "tipo_pregunta": datos_banco.get("tipo_pregunta", ""),
        "complejidad_razonamiento": datos_banco.get("complejidad_razonamiento", ""),
        "distribucion_evidencia": datos_banco.get("distribucion_evidencia", ""),
        "estado": r.get("estado", ""),
        "respuesta_posiblemente_cortada": r.get("respuesta_posiblemente_cortada", ""),
        "tokens_entrada": r.get("tokens_entrada", ""),
        "tokens_salida": r.get("tokens_salida", ""),
        "respuesta_generada": r.get("respuesta_generada", ""),
        "puntuacion_relevancia_respuesta_experta_0_1": "",
        "comentarios_experta": ""
    })

df_experta = pd.DataFrame(filas_experta)

escala_experta = pd.DataFrame([
    {"tramo": "0,0", "interpretacion": "La respuesta no responde a la pregunta o responde de forma incorrecta."},
    {"tramo": "0,1-0,3", "interpretacion": "La respuesta es muy deficiente, demasiado vaga o apenas relacionada con la consulta."},
    {"tramo": "0,4-0,6", "interpretacion": "La respuesta es parcialmente adecuada, pero incompleta, poco precisa o con limitaciones importantes."},
    {"tramo": "0,7-0,8", "interpretacion": "La respuesta es buena y responde a la pregunta, aunque presenta pequeños fallos, imprecisiones u omisiones."},
    {"tramo": "0,9-1,0", "interpretacion": "La respuesta es muy buena o prácticamente perfecta, correcta, suficiente y adecuada al dominio."},
])

# Guardar en Excel para facilitar la revisión y anotación manual
RUTA_EVALUACION_EXPERTA = os.path.join(
    RUTA_BASE,
    "evaluacion_experta_rag_grafos.xlsx"
)

with pd.ExcelWriter(RUTA_EVALUACION_EXPERTA, engine="openpyxl") as writer:
    df_experta.to_excel(
        writer,
        sheet_name="Evaluacion",
        index=False
    )
    escala_experta.to_excel(
        writer,
        sheet_name="Escala",
        index=False
    )

print("Archivo para la experta guardado en:")
print(RUTA_EVALUACION_EXPERTA)

# Mostrar una vista previa
df_experta.head(10)


Archivo para la experta guardado en:
/content/evaluacion_experta_rag_grafos.xlsx


,id,pregunta,tipo_pregunta,complejidad_razonamiento,distribucion_evidencia,estado,respuesta_posiblemente_cortada,tokens_entrada,tokens_salida,respuesta_generada,puntuacion_relevancia_respuesta_experta_0_1,comentarios_experta
0,Q001,¿Qué es el lavado de dinero?,factual simple,single-hop,documento único,ok,False,5090,119,El lavado de dinero es el proceso de cambiar g...,,
1,Q002,¿Qué es el pitufeo?,factual simple,single-hop,documento único,ok,False,5097,100,El pitufeo es una práctica de lavado de activo...,,
2,Q003,¿Qué significa persona expuesta políticamente?,factual simple,single-hop,documento único,ok,False,5078,67,Una persona expuesta políticamente (PEP) es un...,,
3,Q004,¿Qué es la diligencia debida?,factual simple,single-hop,documento único,ok,False,5392,94,La diligencia debida es un proceso de investig...,,
4,Q005,La entidad Amroggang Development Banking Corpo...,factual simple,single-hop,múltiples documentos,ok,False,5436,267,Para determinar si la entidad Amroggang Develo...,,
5,Q006,¿Cuál es la diferencia entre evasión fiscal y ...,comparativa,single-hop,múltiples documentos,ok,False,5209,279,La evasión fiscal y la elusión fiscal son dos ...,,
6,Q007,¿En qué se diferencian una empresa fantasma y ...,comparativa,single-hop,múltiples documentos,ok,False,4932,330,Una empresa fantasma y una organización fachad...,,
7,Q008,¿Qué diferencia existe entre dinero negro y ec...,comparativa,single-hop,múltiples documentos,ok,False,4974,197,La economía sumergida es el conjunto de activi...,,
8,Q009,¿Qué relación existe entre las personas expues...,relacional,multi-hop,múltiples documentos,ok,False,5567,277,"Según el contexto proporcionado, las personas ...",,
9,Q010,¿Cómo se relaciona la propiedad efectiva con e...,relacional,multi-hop,múltiples documentos,ok,False,5069,374,La propiedad efectiva y el cumplimiento normat...,,


In [57]:
from docx import Document
from docx.shared import Pt
import os

RUTA_WORD_EXPERTA = os.path.join(
    RUTA_BASE,
    "evaluacion_experta_rag_grafos.docx"
)

documento = Document()

documento.add_heading("Evaluación experta del sistema RAG de grafos", level=1)

documento.add_paragraph(
    "Este documento recoge las preguntas del banco de evaluación y las respuestas generadas "
    "por el sistema RAG de grafos. La experta valorará la relevancia de la respuesta, es decir, "
    "si la respuesta responde de forma correcta, suficiente y adecuada a la pregunta planteada. "
    "La puntuación debe expresarse entre 0,0 y 1,0, con un decimal."
)

documento.add_heading("Escala de relevancia de la respuesta", level=2)
documento.add_paragraph("0,0: la respuesta no responde a la pregunta o responde de forma incorrecta.")
documento.add_paragraph("0,1-0,3: la respuesta es muy deficiente, demasiado vaga o apenas relacionada con la consulta.")
documento.add_paragraph("0,4-0,6: la respuesta es parcialmente adecuada, pero incompleta, poco precisa o con limitaciones importantes.")
documento.add_paragraph("0,7-0,8: la respuesta es buena y responde a la pregunta, aunque presenta pequeños fallos, imprecisiones u omisiones.")
documento.add_paragraph("0,9-1,0: la respuesta es muy buena o prácticamente perfecta, correcta, suficiente y adecuada al dominio.")

def valor_seguro(fila, columna):
    return str(fila[columna]) if columna in fila and pd.notna(fila[columna]) else ""

for _, fila in df_experta.iterrows():
    documento.add_heading(f"{fila['id']}. {fila['pregunta']}", level=2)

    if valor_seguro(fila, "tipo_pregunta") or valor_seguro(fila, "complejidad_razonamiento") or valor_seguro(fila, "distribucion_evidencia"):
        documento.add_paragraph(
            f"Tipo: {valor_seguro(fila, 'tipo_pregunta')} | "
            f"Complejidad: {valor_seguro(fila, 'complejidad_razonamiento')} | "
            f"Evidencia: {valor_seguro(fila, 'distribucion_evidencia')}"
        )

    if valor_seguro(fila, "respuesta_posiblemente_cortada") == "True":
        documento.add_paragraph(
            "Aviso técnico: esta respuesta aparece marcada como posiblemente cortada y debe revisarse antes de enviarse a evaluación experta."
        )

    p = documento.add_paragraph()
    p.add_run("Respuesta generada:\n").bold = True
    p.add_run(str(fila["respuesta_generada"]))

    documento.add_paragraph("Puntuación de relevancia de la respuesta (0,0-1,0): ____________________")
    documento.add_paragraph("Comentarios de la experta:")
    documento.add_paragraph("______________________________________________________________________")
    documento.add_paragraph("______________________________________________________________________")
    documento.add_paragraph("______________________________________________________________________")

    documento.add_page_break()

documento.save(RUTA_WORD_EXPERTA)

print("Word guardado en:")
print(RUTA_WORD_EXPERTA)


Word guardado en:
/content/evaluacion_experta_rag_grafos.docx


In [58]:
# Descargar archivos de salida desde Colab

from google.colab import files

archivos_salida = [
    RUTA_SALIDA_JSON,
    RUTA_SALIDA_JSON_METRICAS,
    RUTA_SALIDA_CSV_METRICAS,
    RUTA_SALIDA_CSV_RESUMIDO,
    RUTA_EVALUACION_EXPERTA,
    RUTA_WORD_EXPERTA,
    RUTA_UNIDADES_GRAFO,
    RUTA_INDICE_GRAFO
]

for ruta in archivos_salida:
    if os.path.exists(ruta):
        print("Descargando:", ruta)
        files.download(ruta)
    else:
        print("No existe todavía:", ruta)

Descargando: /content/resultados_rag_grafos.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargando: /content/resultados_rag_grafos_con_evaluacion_automatica.json


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargando: /content/indice_faiss_grafo.index


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>